In [1]:
import requests 

docs_url = 'https://github.com/DataTalksClub/llm-zoomcamp/blob/main/01-intro/documents.json?raw=1'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

In [2]:
import hashlib

def generate_document_id(doc):
    # combined = f"{doc['course']}-{doc['question']}"
    combined = f"{doc['course']}-{doc['question']}-{doc['text'][:10]}"
    hash_object = hashlib.md5(combined.encode())
    hash_hex = hash_object.hexdigest()
    document_id = hash_hex[:8]
    return document_id

In [3]:
for doc in documents:
    doc['id'] = generate_document_id(doc)

In [4]:
documents[3]

{'text': "You don't need it. You're accepted. You can also just start learning and submitting homework without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.",
 'section': 'General course-related questions',
 'question': 'Course - I have registered for the Data Engineering Bootcamp. When can I expect to receive the confirmation email?',
 'course': 'data-engineering-zoomcamp',
 'id': '0bbf41ec'}

In [5]:
from collections import defaultdict

In [6]:
hashes = defaultdict(list)

for doc in documents:
    doc_id = doc['id']
    hashes[doc_id].append(doc)

In [7]:
len(hashes), len(documents)

(947, 948)

In [8]:
for k, values in hashes.items():
    if len(values) > 1:
        print(k, len(values))
        

593f7569 2


In [9]:
hashes['593f7569']

[{'text': "They both do the same, it's just less typing from the script.\nAsked by Andrew Katoch, Added by Edidiong Esu",
  'section': '6. Decision Trees and Ensemble Learning',
  'question': 'Does it matter if we let the Python file create the server or if we run gunicorn directly?',
  'course': 'machine-learning-zoomcamp',
  'id': '593f7569'},
 {'text': "They both do the same, it's just less typing from the script.",
  'section': '6. Decision Trees and Ensemble Learning',
  'question': 'Does it matter if we let the Python file create the server or if we run gunicorn directly?',
  'course': 'machine-learning-zoomcamp',
  'id': '593f7569'}]

In [10]:
import json
with open('documents-with-ids.json', 'wt') as f_out:
    json.dump(documents, f_out, indent=2)

In [11]:
prompt_template = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record. 

The record:

section: {section}
question: {question}
answer: {text}

Provide the output in parsable JSON without using code blocks:

["question1", "question2", ..., "question5"]
""".strip()

In [12]:
import sys
import os

# Get the parent directory of the current notebook
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
print(parent_dir)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

print(sys.path)
from api_call import chat_with_openrouter

/home/ubuser/llm-zoomcamp
['/home/ubuser/llm-zoomcamp', '/home/ubuser/spark/spark-3.3.2-bin-hadoop3/python/lib/py4j-0.10.9.5-src.zip', '/home/ubuser/spark/spark-3.3.2-bin-hadoop3/python', '/home/ubuser/llm-zoomcamp/03-evaluation', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/home/ubuser/.local/lib/python3.10/site-packages', '/usr/local/lib/python3.10/dist-packages', '/usr/local/lib/python3.10/dist-packages/eflomal-0.1-py3.10-linux-x86_64.egg', '/usr/lib/python3/dist-packages', '/usr/lib/python3.10/dist-packages']


In [13]:
doc2 = documents[2]
prompt = prompt_template.format(**doc2)
prompt

'You emulate a student who\'s taking our course.\nFormulate 5 questions this student might ask based on a FAQ record. The record\nshould contain the answer to the questions, and the questions should be complete and not too short.\nIf possible, use as fewer words as possible from the record. \n\nThe record:\n\nsection: General course-related questions\nquestion: Course - Can I still join the course after the start date?\nanswer: Yes, even if you don\'t register, you\'re still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don\'t leave everything for the last minute.\n\nProvide the output in parsable JSON without using code blocks:\n\n["question1", "question2", ..., "question5"]'

In [23]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)
    response = chat_with_openrouter(prompt)
    if 'choices' in response and response['choices']:
        return response['choices'][0]['message']['content']
    else:
        print(f"Malformed response for doc {doc.get('id', 'unknown')}: {response}")
        return {'error': 'Malformed response: missing choices'}

In [17]:
res1 = generate_questions(doc2)

In [18]:
print(res1)

[
  "Am I allowed to enroll in this course after its official start date?",
  "Is formal registration required to submit homework assignments?",
  "What deadlines apply to final projects for late-joining students?",
  "Can I successfully complete course requirements if I begin after the scheduled start?",
  "Why is procrastination particularly risky for final project submissions?"
]


In [19]:
from tqdm.auto import tqdm

In [ ]:
#results = {}

In [ ]:
for doc in tqdm(documents): 
    doc_id = doc['id']
    if doc_id in results:
        continue

    questions = generate_questions(doc)
    results[doc_id] = questions

In [29]:
import pickle
# with open('results.bin', 'wb') as f_out:
#     pickle.dump(results, f_out)
with open('results.bin', 'rb') as f_in:
   results = pickle.load(f_in)
#     first_bytes = f_in.read(20)
# print(first_bytes)

In [33]:
print(results['1f6520ca'])

["Where can I find the prerequisites for this course?", "How do I check the prerequisites for this course?", "Where are the course prerequisites listed?", "What are the requirements for joining this course?", "Where is the list of prerequisites for the course?"]


In [43]:
import re
import json

def fix_backslashes(s):
    # Replace any single backslash not already doubled with double backslash
    return re.sub(r'(?<!\\)\\(?![\\"])', r'\\\\', s)

parsed_results = {}
for doc_id, json_questions in results.items():
    if isinstance(json_questions, str):
        try:
            parsed_results[doc_id] = json.loads(json_questions)
        except json.JSONDecodeError as e:
            # Try to fix by escaping backslashes
            fixed = fix_backslashes(json_questions)
            try:
                parsed_results[doc_id] = json.loads(fixed)
            except Exception as e2:
                print(f"Failed to decode {doc_id}:")
                print("First error:", e)
                print("Second error after fix:", e2)
                print("Problematic string:", json_questions)
                # as fallback, assign as string
                parsed_results[doc_id] = json_questions
    else:
        parsed_results[doc_id] = json_questions

Failed to decode 58c9f99f:
First error: Invalid \escape: line 6 column 59 (char 414)
Second error after fix: Invalid \escape: line 6 column 59 (char 414)
Problematic string: [
"How can I resolve the Docker error 'invalid mode: \\Program Files\\Git\\var\\lib\\postgresql\\data'?",
"What should I do if I encounter an invalid mode error in Docker on Windows?",
"What is the correct mounting path to use in Docker for PostgreSQL data on Windows?",
"Can you provide an example of a correct Docker mounting path for PostgreSQL data?",
"How do I correct the mounting path error in Docker for \\\Program Files\\Git\\var\\lib\\postgresql\\data'?"
]


In [44]:
parsed_results['1f6520ca']

['Where can I find the prerequisites for this course?',
 'How do I check the prerequisites for this course?',
 'Where are the course prerequisites listed?',
 'What are the requirements for joining this course?',
 'Where is the list of prerequisites for the course?']

In [45]:
doc_index = {d['id']: d for d in documents}

In [46]:
final_results = []

for doc_id, questions in parsed_results.items():
    course = doc_index[doc_id]['course']
    for q in questions:
        final_results.append((q, course, doc_id))

In [47]:
final_results[0]

('When does the course begin?', 'data-engineering-zoomcamp', 'c02e79ef')

In [48]:
import pandas as pd

In [49]:
df = pd.DataFrame(final_results, columns=['question', 'course', 'document'])

In [50]:
df.head()

,question,course,document
0,When does the course begin?,data-engineering-zoomcamp,c02e79ef
1,How can I get the course schedule?,data-engineering-zoomcamp,c02e79ef
2,What is the link for course registration?,data-engineering-zoomcamp,c02e79ef
3,How can I receive course announcements?,data-engineering-zoomcamp,c02e79ef
4,Where do I join the Slack channel?,data-engineering-zoomcamp,c02e79ef


In [51]:
df.to_csv('ground-truth-data.csv', index=False)

In [52]:
!head ground-truth-data.csv

question,course,document
When does the course begin?,data-engineering-zoomcamp,c02e79ef
How can I get the course schedule?,data-engineering-zoomcamp,c02e79ef
What is the link for course registration?,data-engineering-zoomcamp,c02e79ef
How can I receive course announcements?,data-engineering-zoomcamp,c02e79ef
Where do I join the Slack channel?,data-engineering-zoomcamp,c02e79ef
Where can I find the prerequisites for this course?,data-engineering-zoomcamp,1f6520ca
How do I check the prerequisites for this course?,data-engineering-zoomcamp,1f6520ca
Where are the course prerequisites listed?,data-engineering-zoomcamp,1f6520ca
What are the requirements for joining this course?,data-engineering-zoomcamp,1f6520ca
